# A worker goroutine that outlives the cell that started it

This notebook starts a background worker goroutine in one cell, then feeds it jobs and
collects its results from separate, later cells, and finally shuts it down. Between cell
executions the worker is genuinely running in the background -- not paused, not
re-simulated, the same live goroutine the whole time.

This only works because gosk keeps every cell's compiled plugin loaded in the *same*
long-lived kernel process: a `go func() { ... }()` started in cell 1 is scheduled by that
process's Go runtime and keeps running for as long as the process does, regardless of which
cell is currently executing (or whether any cell is executing at all). A kernel design that
relaunches a process per cell has nothing to attach a background goroutine to between
executions -- there's no process left for it to run in once the cell that started it returns.

In [ ]:
jobs := make(chan int, 100)
results := make(chan int, 100)
stop := make(chan struct{})

go func() {
	for {
		select {
		case n := <-jobs:
			sum := 0
			for i := 0; i < n; i++ {
				sum += i
			}
			results <- sum
		case <-stop:
			return
		}
	}
}()

fmt.Println("Background worker started")

## Submit work from a later, independent cell

The worker is already running -- this cell only sends values into `jobs`.

In [ ]:
jobs <- 1000000
jobs <- 2000000
fmt.Println("Jobs submitted")

## Collect the results, whenever you get around to it

There's no deadline tying this cell to the one before it -- the worker just keeps whatever
it computed sitting in the buffered `results` channel until something reads it.

In [ ]:
r1 := <-results
r2 := <-results
fmt.Println("Results:", r1, r2)

In [ ]:
// Auto-displayed as this cell's result, the equivalent of Jupyter's Out[n].
r1 + r2

## Shut the worker down

Closing `stop` releases the worker's `select` and lets its goroutine return.

In [ ]:
close(stop)
fmt.Println("Background worker stopped")